# Med-DDPM v3.1 — Fine-Tuning from Epoch 90
Continue training from the best checkpoint with:
- Lower LR (5e-5) → less overfitting
- All data (train+val+test merged, 80/20 balanced) → better generalization
- 30 more epochs → fine-tuning, not from scratch

## 1. Colab Setup & GPU Check

In [ ]:
import os
import torch

COLAB_WORKING = "/content"
DRIVE_DIR = "/content/drive/MyDrive/mask-to-mri"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("⚠️  No GPU detected!")

## 2. Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = "/content/Mask-to-MRI"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AmineAitLaamim/Mask-to-MRI.git
    print("✅ Repository cloned.")
else:
    %cd $REPO_DIR
    !git pull
    print("✅ Repository updated.")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

## 3. Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations tifffile scikit-image pyyaml tqdm matplotlib
print("Dependencies installed")

## 4. Create Output Directories

In [ ]:
OUTPUTS_DIR = "/content/Mask-to-MRI/outputs_v3_1"
for d in ["checkpoints", "samples", "metrics", "synthetic"]:
    os.makedirs(f"{OUTPUTS_DIR}/{d}", exist_ok=True)
    print(f"Created {OUTPUTS_DIR}/{d}")

## 5. Upload Dataset

In [ ]:
import shutil

RAW_DIR = "/content/Mask-to-MRI/dataset/lgg-mri-segmentation"
if not os.path.exists(RAW_DIR):
    zip_in_drive = f"{DRIVE_DIR}/dataset/lgg-mri-segmentation.zip"
    if os.path.exists(zip_in_drive):
        print(f"Found zip at: {zip_in_drive}")
        shutil.unpack_archive(zip_in_drive, "/content/Mask-to-MRI/dataset")
        print("✅ Dataset extracted.")
    else:
        print("⚠️  Dataset not found. Upload lgg-mri-segmentation.zip to:")
        print(f"   {DRIVE_DIR}/dataset/")
else:
    print(f"✅ Dataset already at: {RAW_DIR}")

print(f"RAW_DIR: {RAW_DIR}")

## 6. Import Modules & Load Config

In [ ]:
import sys
for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, '/content/Mask-to-MRI')

from src.med_ddpm_v3_1 import ConditionalDDPM, CONFIG
from src.med_ddpm_v3_1.train import train
from src.med_ddpm_v3_1.utils import build_all_data_flair_loader

# Override paths with absolute Colab paths
CONFIG["raw_dir"] = RAW_DIR
CONFIG["checkpoint_dir"] = f"{OUTPUTS_DIR}/checkpoints"
CONFIG["samples_dir"] = f"{OUTPUTS_DIR}/samples"
CONFIG["metrics_dir"] = f"{OUTPUTS_DIR}/metrics"
CONFIG["synthetic_dir"] = f"{OUTPUTS_DIR}/synthetic"

print("Config loaded:")
print(f"  raw_dir: {CONFIG['raw_dir']}")
print(f"  LR: {CONFIG['lr']}")
print(f"  Epochs (additional): {CONFIG['epochs']}")
print(f"  Dropout: {CONFIG['dropout']}")
print(f"  EMA decay: {CONFIG['ema_decay']}")
print(f"  Resume from: {CONFIG.get('resume_from', 'None')}")

## 7. Device Setup & Seeds

In [ ]:
import random
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("TF32 enabled")

seed = CONFIG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
print(f"Random seed fixed: {seed}")

## 8. Build All-Data FLAIR Dataloader

In [ ]:
loader = build_all_data_flair_loader(
    CONFIG["raw_dir"],
    batch_size=CONFIG["batch_size"],
    num_workers=4,
    tumor_ratio=CONFIG["tumor_ratio"],
    seed=CONFIG["seed"],
)

loader.pin_memory = torch.cuda.is_available()

print(f"All-data training loader:")
print(f"  Total: {len(loader.dataset)} samples")
print(f"  Batches/epoch: {len(loader)}")

## 9. Verify Batch Shape

In [ ]:
mask_batch, flair_batch = next(iter(loader))
print(f"Batch shape: mask={mask_batch.shape}, flair={flair_batch.shape}")
print(f"FLAIR mean: {flair_batch[:,0].mean():.3f}, std: {flair_batch[:,0].std():.3f}")

assert flair_batch.shape[1] == 1, f"Expected 1 channel, got {flair_batch.shape[1]}"
print("✅ Single-channel FLAIR verified")

## 10. Create Model

In [ ]:
model = ConditionalDDPM(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} parameters ({n_params/1e6:.2f}M)")

## 11. Pre-Training Sanity Check

In [ ]:
print("=" * 60)
print("PRE-TRAINING SANITY CHECK")
print("=" * 60)

mask_d = mask_batch[:2].to(device)
flair_d = flair_batch[:2].to(device)
loss = model(flair_d, mask_d)
print(f"Forward pass OK — loss={loss.item():.4f}")

import time
start = time.time()
with torch.no_grad():
    fake = model.sample(mask_d[:1], ddim_steps=10)
elapsed = time.time() - start
print(f"Sample (10 DDIM steps): shape={fake.shape}, std={fake.std():.3f}")
print(f"10 steps took: {elapsed:.1f}s")
print(f"Full 250 steps estimated: {elapsed * 25:.0f}s per image")

print("=" * 60)
print("✅ SANITY CHECK COMPLETE — ready to fine-tune")
print("=" * 60)

## 12. Restore Checkpoint from Drive (after session restart)

In [ ]:
import glob, os, shutil
from pathlib import Path

drive_ckpt_dir = f"{DRIVE_DIR}/outputs_v3/checkpoints"
local_ckpt_dir = CONFIG["checkpoint_dir"]

drive_ckpts = glob.glob(f"{drive_ckpt_dir}/checkpoint_v3_epoch_*.pt")
local_ckpts = glob.glob(f"{local_ckpt_dir}/checkpoint_v3_1_epoch_*.pt")

if not local_ckpts and drive_ckpts:
    latest = max(drive_ckpts)
    os.makedirs(local_ckpt_dir, exist_ok=True)
    shutil.copy2(latest, local_ckpt_dir)
    print(f"Restored from Drive: {os.path.basename(latest)}")
elif local_ckpts:
    print(f"Found {len(local_ckpts)} local checkpoint(s)")
else:
    print("No local checkpoints, checking Drive...")

if drive_ckpts:
    print(f"Drive has {len(drive_ckpts)} checkpoint(s)")
else:
    print("No checkpoints on Drive.")

## 13. Auto-Resume from v3 Epoch 90 Checkpoint

In [ ]:
resume_from = CONFIG.get("resume_from")
if resume_from and Path(resume_from).exists():
    print(f"Found checkpoint: {resume_from}")
    print("   Fine-tuning will resume from v3 epoch 90")
else:
    print("No v3 checkpoint found — starting from scratch.")
    resume_from = None

## 14. Start Fine-Tuning

In [ ]:
history = train(
    train_loader=loader,
    val_loader=loader,
    model=model,
    config=CONFIG,
    device=device,
    resume_from=resume_from,
)

## 15. Plot Training Loss

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = f"{CONFIG['metrics_dir']}/v3_1_training_history.json"
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    epochs = [h["epoch"] for h in history]
    losses = [h["loss"] for h in history]
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, losses, "b-", linewidth=2, label="Train")
    if "val_loss" in history[0]:
        val_losses = [h["val_loss"] for h in history]
        plt.plot(epochs, val_losses, "r-", linewidth=2, label="Val")
        plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("L1 Loss (Min-SNR weighted)")
    plt.title("DDPM v3.1 Fine-Tuning Loss")
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No training history found yet.")

## 16. Generate Synthetic FLAIR Images

In [ ]:
from src.med_ddpm_v3_1.sample import generate_synthetic

checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint_v3_1_epoch_*.pt")
if checkpoints:
    best = max(checkpoints, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Generating from: {best}")
    generate_synthetic(
        checkpoint_path=best,
        output_dir=CONFIG["synthetic_dir"],
        config=CONFIG,
        cfg_scale=1.0,
    )
else:
    print("No checkpoints found. Train the model first.")